In [2]:
# CELL 1 — Imports

import torch
import timm
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from torchvision import datasets
from torch.utils.data import DataLoader, random_split
from sklearn.metrics import accuracy_score, classification_report
import matplotlib.pyplot as plt
import os

print("✅ All imports done")
print(f"PyTorch version: {torch.__version__}")
print(f"GPU available: {torch.cuda.is_available()}")

✅ All imports done
PyTorch version: 2.5.1+cu121
GPU available: True


In [3]:
# CELL 2 — PATHS (based on your actual folder)

from pathlib import Path

# Repo root: walk up from the notebook's directory until app.py is found.
BASE_PATH    = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "app.py").is_file())
KAGGLE_TRAIN = BASE_PATH / "datasets" / "KAGGLE DATASET" / "Combined Dataset" / "train"
MODELS_PATH  = BASE_PATH / "models"
RESULTS_PATH = BASE_PATH / "results"

# Create folders if they don't exist
os.makedirs(MODELS_PATH,  exist_ok=True)
os.makedirs(RESULTS_PATH, exist_ok=True)

# Verify paths exist
print("Checking paths:")
for name, path in [("Train data", KAGGLE_TRAIN),
                   ("Models",     MODELS_PATH),
                   ("Results",    RESULTS_PATH)]:
    status = "✅" if os.path.exists(path) else "❌ NOT FOUND"
    print(f"  {status} {name}: {path}")

Checking paths:
  ✅ Train data: datasets\KAGGLE DATASET\Combined Dataset\train
  ✅ Models: models
  ✅ Results: results


In [4]:
# CELL 3 — Load and split data

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225])
])

# Load full dataset
full_dataset = datasets.ImageFolder(
    root=KAGGLE_TRAIN,
    transform=train_transform
)

# Print what was found
print(f"Classes found: {full_dataset.classes}")
print(f"Total images:  {len(full_dataset)}")

# Split 70/15/15
total      = len(full_dataset)
train_size = int(0.70 * total)
val_size   = int(0.15 * total)
test_size  = total - train_size - val_size

train_data, val_data, test_data = random_split(
    full_dataset,
    [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

# Create loaders
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_data,   batch_size=32, shuffle=False)
test_loader  = DataLoader(test_data,  batch_size=32, shuffle=False)

print(f"\nTrain: {train_size} images")
print(f"Val:   {val_size} images")
print(f"Test:  {test_size} images")
print(f"\nTrain batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")
print(f"Test batches:  {len(test_loader)}")

Classes found: ['Mild Impairment', 'Moderate Impairment', 'No Impairment', 'Very Mild Impairment']
Total images:  10240

Train: 7168 images
Val:   1536 images
Test:  1536 images

Train batches: 224
Val batches:   48
Test batches:  48


In [5]:
device = torch.device('cuda' if torch.cuda.is_available() 
                       else 'cpu')

def build_vit():
    model = timm.create_model(
        'vit_base_patch16_224',
        pretrained=True,
        num_classes=4
    )
    
    # Freeze all except last 3 blocks + head
    for name, param in model.named_parameters():
        if any(x in name for x in
               ['blocks.9', 'blocks.10', 'blocks.11',
                'head', 'norm']):
            param.requires_grad = True
        else:
            param.requires_grad = False
    
    return model

vit_model_test = build_vit()
total     = sum(p.numel() for p in vit_model_test.parameters())
trainable = sum(p.numel() for p in vit_model_test.parameters()
                if p.requires_grad)

print(f"✅ Vision Transformer built")
print(f"Total params:     {total:,}")
print(f"Trainable params: {trainable:,}")

✅ Vision Transformer built
Total params:     85,801,732
Trainable params: 21,295,876


In [6]:
def train_vit_and_save(model, model_name, epochs=20):
    
    model = model.to(device)
    
    # ViT uses 0.0001 NOT 0.001
    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=0.0001
    )
    criterion = nn.CrossEntropyLoss()
    
    # Cosine scheduler for ViT
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=epochs
    )
    
    best_val_acc = 0
    train_accs   = []
    val_accs     = []
    
    print(f"Training {model_name} for {epochs} epochs...")
    print("=" * 55)
    
    for epoch in range(epochs):
        
        model.train()
        correct = 0
        total   = 0
        
        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss    = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            _, preds = outputs.max(1)
            correct += preds.eq(labels).sum().item()
            total   += labels.size(0)
        
        train_acc = 100 * correct / total
        
        model.eval()
        correct = 0
        total   = 0
        
        with torch.no_grad():
            for images, labels in val_loader:
                images  = images.to(device)
                labels  = labels.to(device)
                outputs = model(images)
                _, preds = outputs.max(1)
                correct += preds.eq(labels).sum().item()
                total   += labels.size(0)
        
        val_acc = 100 * correct / total
        scheduler.step()
        
        train_accs.append(train_acc)
        val_accs.append(val_acc)
        
        saved = ""
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            save_path    = os.path.join(MODELS_PATH,
                                        f'{model_name}_best.pth')
            torch.save(model.state_dict(), save_path)
            saved = " ← SAVED ✅"
        
        print(f"Epoch {epoch+1:02d}/{epochs} | "
              f"Train: {train_acc:.1f}% | "
              f"Val: {val_acc:.1f}%{saved}")
    
    print("=" * 55)
    print(f"Best Val Accuracy: {best_val_acc:.2f}%")
    return model, train_accs, val_accs, best_val_acc

In [7]:
vit_model, train_accs, val_accs, best_acc = train_vit_and_save(
    model=build_vit(),
    model_name='vit',
    epochs=20
)
print(f"✅ Vision Transformer COMPLETE: {best_acc:.2f}%")

Training vit for 20 epochs...
Epoch 01/20 | Train: 66.6% | Val: 74.5% ← SAVED ✅
Epoch 02/20 | Train: 80.6% | Val: 82.9% ← SAVED ✅
Epoch 03/20 | Train: 83.4% | Val: 84.7% ← SAVED ✅
Epoch 04/20 | Train: 87.4% | Val: 88.0% ← SAVED ✅
Epoch 05/20 | Train: 89.2% | Val: 86.3%
Epoch 06/20 | Train: 90.9% | Val: 88.7% ← SAVED ✅
Epoch 07/20 | Train: 94.0% | Val: 91.0% ← SAVED ✅
Epoch 08/20 | Train: 94.2% | Val: 92.1% ← SAVED ✅
Epoch 09/20 | Train: 96.3% | Val: 92.2% ← SAVED ✅
Epoch 10/20 | Train: 96.8% | Val: 94.4% ← SAVED ✅
Epoch 11/20 | Train: 97.7% | Val: 94.1%
Epoch 12/20 | Train: 98.3% | Val: 95.8% ← SAVED ✅
Epoch 13/20 | Train: 98.9% | Val: 94.5%
Epoch 14/20 | Train: 98.9% | Val: 95.5%
Epoch 15/20 | Train: 99.4% | Val: 95.8%
Epoch 16/20 | Train: 99.4% | Val: 96.2% ← SAVED ✅
Epoch 17/20 | Train: 99.5% | Val: 96.7% ← SAVED ✅
Epoch 18/20 | Train: 99.8% | Val: 96.7%
Epoch 19/20 | Train: 99.8% | Val: 96.7% ← SAVED ✅
Epoch 20/20 | Train: 99.8% | Val: 97.6% ← SAVED ✅
Best Val Accuracy: 97.59%
✅ Vi